# Superhero Data Analysis with Pandas

## Business Context

This project uses a modified superhero dataset to practice a realistic pandas workflow: loading data, cleaning missing and invalid values, reshaping, joining, grouping, and visualizing results.

The analysis answers three business questions:

1. What is the distribution of superheroes by publisher?
2. What is the relationship between height and number of superpowers, and does it differ by gender?
3. What are the five most common superpowers in Marvel Comics vs. DC Comics?

A final exploratory section also asks an additional question of the data.

## Objectives

- Load and inspect CSV data with pandas
- Handle missing and invalid values
- Normalize inconsistent text values
- Reshape DataFrames with transpose and reset_index
- Merge multiple datasets
- Aggregate grouped data
- Create visualizations that answer business questions

In [ ]:
# CodeGrade step0
# Run this cell without any changes

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Section 1 — Load and Inspect the Data

### Step 1 — Heroes Data
Load `heroes_information.csv` using the first column as the index, then inspect its shape.

In [ ]:
# CodeGrade step1
heroes_df = pd.read_csv("heroes_information.csv", index_col=0)
heroes_shape = heroes_df.shape
heroes_shape

In [ ]:
# Run this cell without any changes
heroes_df.info()

### Step 2 — Superpowers Data
Load `super_hero_powers.csv` using the first column as the index. The index contains the superpower names, which we will need later when reshaping and joining.

In [ ]:
# CodeGrade step2
powers_df = pd.read_csv("super_hero_powers.csv", index_col=0)
powers_shape = powers_df.shape
powers_shape

In [ ]:
# Run this cell without changes
powers_df.info()

## Section 2 — Publisher Distribution

### Business Question 1
**What is the distribution of superheroes by publisher?**

Before answering, we need to remove missing publisher values and normalize inconsistent publisher names.

### Step 3 — Handle Missing Publisher Values
Only rows missing `Publisher` should be removed. Other missing fields should remain untouched because they may still be useful for later questions.

In [ ]:
# CodeGrade step3
heroes_df = heroes_df.dropna(subset=["Publisher"])

In [ ]:
# Run this cell without changes
heroes_df.info()

### Step 4 — Inspect Publisher Labels
Use `value_counts()` to identify duplicate publisher labels caused by whitespace or naming inconsistencies.

In [ ]:
# CodeGrade step4
pub_counts = heroes_df["Publisher"].value_counts()
pub_counts

### Step 5 — Normalize Publisher Names
Strip surrounding whitespace and standardize the two largest publishers as `Marvel Comics` and `DC Comics`.

In [ ]:
# CodeGrade step5
heroes_df["Publisher"] = heroes_df["Publisher"].str.strip()
heroes_df["Publisher"] = heroes_df["Publisher"].replace({"Marvel": "Marvel Comics", "DC": "DC Comics"})
correct_counts = heroes_df["Publisher"].value_counts()
correct_counts

In [ ]:
# Distribution of superheroes by publisher
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(16, 5))
value_counts = heroes_df["Publisher"].value_counts()
top_5_counts = value_counts.iloc[:5]
ax1.bar(value_counts.index, value_counts.values)
ax2.bar(top_5_counts.index, top_5_counts.values)
ax1.tick_params(axis="x", labelrotation=90)
ax2.tick_params(axis="x", labelrotation=45)
ax1.set_ylabel("Count of Superheroes")
ax2.set_ylabel("Count of Superheroes")
ax1.set_title("Distribution of Superheroes by Publisher")
ax2.set_title("Top 5 Publishers by Count of Superheroes");

### Analyst Interpretation
The publisher distribution is highly concentrated in the largest comic publishers, so normalizing publisher names is essential before comparing counts. Even small text inconsistencies would otherwise split one publisher into multiple categories.

## Section 3 — Height and Number of Superpowers

### Business Question 2
**What is the relationship between height and number of superpowers, and does this differ based on gender?**

This question requires combining the hero attributes and power data.

### Step 6 — Transpose the Powers DataFrame
The original power table has superpowers as rows and superheroes as columns. Transposing it creates one row per superhero, which is compatible with the hero information table.

In [ ]:
# CodeGrade step6
powers_df_transposed = powers_df.transpose()
powers_df_transposed.head()

### Step 7 — Reset the Index
Rename the current index to `name`, then reset it without dropping the old index so superhero names become a normal column.

In [ ]:
# CodeGrade step7
powers_df_transposed.index.name = "name"
powers_df_transposed = powers_df_transposed.reset_index()
powers_df_transposed.head()

### Step 8 — Merge the DataFrames
Merge the reshaped powers data with hero attributes using the shared `name` field.

In [ ]:
# CodeGrade step8
heroes_and_powers_df = heroes_df.merge(powers_df_transposed, on="name")

In [ ]:
# Calculate total number of powers for each hero
heroes_and_powers_df["Power Count"] = sum([heroes_and_powers_df[power_name] for power_name in powers_df.index])
heroes_and_powers_df

### Step 9 — Remove Invalid Heights
Negative heights are sentinel values for missing data rather than physically meaningful measurements. Create a separate DataFrame containing only heroes with nonnegative heights.

In [ ]:
# CodeGrade step9
question_2_df = heroes_and_powers_df[heroes_and_powers_df["Height"] >= 0].copy()
question_2_df.head()

In [ ]:
# Height vs. number of superpowers by gender
fig, ax = plt.subplots(figsize=(16, 8))
question_2_male = question_2_df[question_2_df["Gender"] == "Male"]
question_2_female = question_2_df[question_2_df["Gender"] == "Female"]
question_2_other = question_2_df[(question_2_df["Gender"] != "Male") & (question_2_df["Gender"] != "Female")]
ax.scatter(question_2_male["Height"], question_2_male["Power Count"], alpha=0.5, label="Male")
ax.scatter(question_2_female["Height"], question_2_female["Power Count"], alpha=0.5, label="Female")
ax.scatter(question_2_other["Height"], question_2_other["Power Count"], alpha=0.5, label="Other")
ax.set_xlabel("Height (cm)")
ax.set_ylabel("Number of Superpowers")
ax.set_title("Height vs. Power Count by Gender")
ax.legend();

### Analyst Interpretation
The scatter plot does not show a strong relationship between height and total number of powers. Separating points by gender does not materially change that conclusion, although the gender groups occupy somewhat different height ranges.

## Section 4 — Most Common Powers in Marvel vs. DC

### Business Question 3
**What are the five most common superpowers in Marvel Comics vs. DC Comics?**

This requires filtering to the two publishers, keeping only publisher and power columns, grouping by publisher, summing Boolean power values, and reshaping the result.

### Step 10 — Filter Publishers and Relevant Columns

In [ ]:
# CodeGrade step10
question_3_df = heroes_and_powers_df.copy()
question_3_df = question_3_df[question_3_df["Publisher"].isin(["Marvel Comics", "DC Comics"])]
relevant_columns = list(powers_df.index)
relevant_columns.append("Publisher")
question_3_df = question_3_df[relevant_columns]
question_3_df.head()

### Step 11 — Group and Sum Powers
Because Boolean values behave as 1 and 0 in numeric aggregation, summing each power column gives the count of heroes with that power for each publisher.

In [ ]:
# CodeGrade step11
question_3_grouped = question_3_df.groupby("Publisher").sum()
question_3_grouped

### Step 12 — Transpose the Grouped Result
Transpose again so each record represents one superpower and publisher counts become columns.

In [ ]:
# CodeGrade step12
question_3_grouped = question_3_grouped.transpose()
question_3_grouped

### Step 13 — Reset the Superpower Index
Rename the index to `Superpower Name` and reset it so the power names become a normal column.

In [ ]:
# CodeGrade step13
question_3_grouped.index.name = "Superpower Name"
question_3_grouped = question_3_grouped.reset_index()
question_3_grouped

In [ ]:
# Find the five most common powers for each publisher
marvel_most_common = question_3_grouped[["Superpower Name", "Marvel Comics"]].sort_values(by="Marvel Comics", ascending=False).head(5)
dc_most_common = question_3_grouped[["Superpower Name", "DC Comics"]].sort_values(by="DC Comics", ascending=False).head(5)

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(15, 5))
ax1.bar(marvel_most_common["Superpower Name"], marvel_most_common["Marvel Comics"])
ax2.bar(dc_most_common["Superpower Name"], dc_most_common["DC Comics"])
ax1.set_ylabel("Count of Superheroes")
ax2.set_ylabel("Count of Superheroes")
ax1.set_title("Top Superpowers in Marvel Comics")
ax2.set_title("Top Superpowers in DC Comics")
plt.setp(ax1.get_xticklabels(), rotation=45, ha="right")
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right");

### Analyst Interpretation
The two publishers share several highly common powers, which suggests substantial overlap in the abilities assigned to their heroes. The ranked counts make it easy to see which abilities dominate each publisher's catalog.

## Section 5 — Exploratory Question

### Question
**Which gender has the highest median number of superpowers?**

This question uses the cleaned merged dataset and combines descriptive statistics with a visualization.

In [ ]:
# Compare median power count by gender
gender_power_medians = question_2_df.groupby("Gender")["Power Count"].median().sort_values(ascending=False)
gender_power_medians

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
gender_power_medians.plot(kind="bar", ax=ax)
ax.set_xlabel("Gender")
ax.set_ylabel("Median Number of Superpowers")
ax.set_title("Median Power Count by Gender");

## Business Summary

This project demonstrates a complete pandas analysis workflow: data loading, missing-value handling, text normalization, reshaping, merging, filtering invalid values, groupby aggregation, and visualization.

The publisher analysis requires careful text cleaning before counts are trustworthy. Height does not appear to have a strong relationship with total power count, even when gender is considered. Marvel and DC share several common high-frequency powers, showing substantial similarity in their most prevalent abilities.

The final exploratory analysis extends the workflow by comparing median power counts across gender categories.